In [6]:
import yfinance as yf
import pandas as pd
import numpy as np
import akshare as ak
import statsmodels.api as sm
import matplotlib.pyplot as plt
import pandas_datareader.data as web

In [9]:
import ssl
import warnings
import urllib3
import requests
from pathlib import Path

In [3]:
# 1. Silenciar las advertencias visuales de conexiones no verificadas
warnings.filterwarnings("ignore")
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# 2. Desactivar verificación en el motor SSL nativo de Python
ssl._create_default_https_context = ssl._create_unverified_context

# 3. Desactivar verificación en requests (corrige FRED, pandas_datareader y akshare)
_old_session_init = requests.Session.__init__
def _insecure_session_init(self, *args, **kwargs):
    _old_session_init(self, *args, **kwargs)
    self.verify = False
requests.Session.__init__ = _insecure_session_init

# 4. Desactivar verificación en curl_cffi (corrige el motor interno de yfinance)
try:
    from curl_cffi import requests as cureq
    _old_cureq_init = cureq.Session.__init__
    def _insecure_cureq_init(self, *args, **kwargs):
        _old_cureq_init(self, *args, **kwargs)
        self.verify = False
    cureq.Session.__init__ = _insecure_cureq_init
except ImportError:
    pass

print("Parche SSL aplicado con éxito. Conexiones liberadas.")

Parche SSL aplicado con éxito. Conexiones liberadas.


In [11]:
# ==============================================================================
# 1. PARÁMETROS TEMPORALES Y TICKERS
# ==============================================================================
fecha_inicio = "2000-01-01"
fecha_fin = "2026-06-01"
ONZAS_POR_GRAMO = 31.1034768

print("1. Descargando mercados de oro occidentales y fundamentales macro...")

# Activos de Oro: GC=F (COMEX), GLD (Proxy LBMA), 518880.SS (ETF China)
# Fundamentales: UUP (Proxy Dólar / DXY), TIP (Tasas Reales TIPS), CNY=X (Tipo de cambio)
tickers_yf = ["GC=F", "GLD", "518880.SS", "UUP", "TIP", "CNY=X"]
df_yf = yf.download(tickers_yf, start=fecha_inicio, end=fecha_fin, auto_adjust=False)['Adj Close']

# Normalizar índice temporal para eliminar zonas horarias
df_yf.index = pd.to_datetime(df_yf.index).tz_localize(None).normalize()

# Separación de series individuales
comex = df_yf['GC=F'].rename("COMEX_Futuros_USD")
usdcny = df_yf['CNY=X'].rename("USD_CNY")
dolar_dxy = df_yf['UUP'].rename("Dolar_DXY")
tasas_reales = df_yf['TIP'].rename("Tasas_Reales_TIPS")

# Reconstruir precio físico de Londres mediante el proxy GLD
ratio_conversion_gld = (comex / df_yf['GLD']).median()
lbma_proxy_usd = (df_yf['GLD'] * ratio_conversion_gld).rename("LBMA_Proxy_Spot_USD")

# ==============================================================================
# 2. DESCARGA DE MERCADOS CHINOS (akshare)
# ==============================================================================
print("2. Descargando series domésticas de China (SGE y SHFE)...")

# Spot SGE (Au99.99) en RMB/g
df_sge = ak.spot_hist_sge(symbol="Au99.99")
df_sge['date'] = pd.to_datetime(df_sge['date']).dt.tz_localize(None).dt.normalize()
sge_rmb = df_sge.set_index('date')['close'].loc[fecha_inicio:fecha_fin].rename("SGE_Au9999_RMB_g")

# Futuros SHFE (au0) en RMB/g
df_shfe = ak.futures_zh_daily_sina(symbol="au0")
df_shfe['date'] = pd.to_datetime(df_shfe['date']).dt.tz_localize(None).dt.normalize()
shfe_rmb = df_shfe.set_index('date')['close'].loc[fecha_inicio:fecha_fin].rename("SHFE_Futuros_RMB_g")

# ==============================================================================
# 3. ALINEACIÓN, CONVERSIÓN Y CALENDARIOS CRUZADOS
# ==============================================================================
print("3. Homogenizando unidades a USD/oz y sincronizando panel...")
panel_precios = pd.concat([comex, lbma_proxy_usd, sge_rmb, shfe_rmb, usdcny, dolar_dxy, tasas_reales], axis=1)

# Normalizar precios de China a USD por onza troy
panel_precios['SGE_Spot_USD_oz'] = (panel_precios['SGE_Au9999_RMB_g'] * ONZAS_POR_GRAMO) / panel_precios['USD_CNY']
panel_precios['SHFE_Futuros_USD_oz'] = (panel_precios['SHFE_Futuros_RMB_g'] * ONZAS_POR_GRAMO) / panel_precios['USD_CNY']


# ==============================================================================
# 4. RETORNOS LOGARÍTMICOS CONTINUOS
# ==============================================================================
# Seleccionar series de precios finales y factores macro
series_analisis = [
    'COMEX_Futuros_USD', 'LBMA_Proxy_Spot_USD', 
    'SGE_Spot_USD_oz', 'SHFE_Futuros_USD_oz',
    'Dolar_DXY', 'Tasas_Reales_TIPS'
]


# Guardar matrices procesadas para modelación en R
panel_precios.to_csv("../data/panel_precios_niveles.csv")

print("\nArchivos exportados exitosamente:")
print("1. data/processed/panel_precios_niveles.csv (Niveles, primas y spreads en USD)")

1. Descargando mercados de oro occidentales y fundamentales macro...


[*********************100%***********************]  6 of 6 completed


2. Descargando series domésticas de China (SGE y SHFE)...
3. Homogenizando unidades a USD/oz y sincronizando panel...

Archivos exportados exitosamente:
1. data/processed/panel_precios_niveles.csv (Niveles, primas y spreads en USD)


In [12]:
panel_precios

,COMEX_Futuros_USD,LBMA_Proxy_Spot_USD,SGE_Au9999_RMB_g,SHFE_Futuros_RMB_g,USD_CNY,Dolar_DXY,Tasas_Reales_TIPS,SGE_Spot_USD_oz,SHFE_Futuros_USD_oz
2000-08-30,273.899994,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-08-31,278.299988,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-09-01,277.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-09-05,275.799988,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-09-06,274.200012,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
2026-05-26,4502.299805,4319.202282,991.97,995.00,6.7945,27.750000,107.697105,4540.984098,4554.854660
2026-05-27,4448.399902,4261.717145,982.53,983.56,6.7860,27.750000,107.755417,4503.404078,4508.125060
2026-05-28,4499.299805,4306.369755,961.82,961.18,6.7788,27.700001,108.046959,4413.162508,4410.225967
2026-05-29,4593.000000,4351.752741,984.96,989.40,6.7795,27.660000,108.076118,4518.870193,4539.240344
